# Objective Pupil and First-Order Filtering

Stage C geometry notebook for the `slm_plane -> fourier_filter_plane -> objective_pupil_plane -> surface_plane` route. It separates objective/NA geometry from full propagation so clipping and first-order limits are visible.


## Stage 8.7 Adjustable Quick-Look Guidance

<!-- STAGE87: adjustable quicklook guidance -->

For fast parameter scouting, use `notebooks/quicklook/00_quick_beam_to_sample_simulator.ipynb`. This notebook remains on its locked stage path: existing execution logic, propagation-power labels, material-proxy caveats, and governance routing are unchanged.

Safe local edits are the explicit config variables already exposed by this notebook, or a copied exploratory run. Keep `fail` and `marginal` labels visible. If a displayed image is visually smoothed, treat that as display interpolation only; rerun balanced/publication sampling before numerical interpretation.


In [1]:
from dataclasses import replace
from pathlib import Path

import pandas as pd

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_regime
from vbb_study.equations import objective_pupil as objp
from vbb_study.publication import lab_realism as lab_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
out_csv = PATHS["csv"] / "stage_c"
out_csv.mkdir(parents=True, exist_ok=True)
base = replace(bt.default_config(PRESET), generation_method="holographic")

def _common(case_id, cfg, plane_label, coordinate_frame, **extra):
    row = {
        "case_id": case_id,
        "objective_NA": float(cfg.objective.NA),
        "objective_f_eff_mm": float(cfg.objective.f_eff_m / bt.mm),
        "pupil_radius_mm": float(cfg.objective.pupil_radius_m / bt.mm),
        **extra,
    }
    lab_schema.annotate_lab_realism_row(
        row,
        generation_method="objective_pupil_limited",
        model_level="hardware_route",
        hardware_status="current_lab_realizable",
        plane_label=plane_label,
        coordinate_frame=coordinate_frame,
        run_id=RUN_ID,
        preset=PRESET,
        path="geometry_only",
    )
    return row


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(stage='lab_realism')
try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [2]:
rows = []
for regime in ("general", "limits"):
    cfg = vbb_regime.config_for_regime(base, regime)
    design = bt.compute_design_from_targets(cfg.laser, cfg.target, cfg.material)
    pupil_radius = objp.pupil_radius_m(cfg.objective.f_eff_m, cfg.objective.NA, cfg.objective.immersion_n)
    fill = objp.gaussian_pupil_fill_fraction(cfg.laser.beam_radius_on_slm_m, pupil_radius)
    rows.append(_common(
        f"{regime}_objective_pupil",
        cfg,
        "objective_pupil_plane",
        "objective_pupil_plane_mm",
        regime=regime,
        pupil_radius_formula="f_eff_m * NA / immersion_n",
        pupil_fill_fraction=fill,
        pupil_clipped_fraction=1.0 - fill,
        pupil_fill_ratio=objp.pupil_fill_ratio(cfg.laser.beam_radius_on_slm_m, pupil_radius),
        fourier_ring_radius_mm=objp.fourier_plane_ring_radius_m(design.kr_slm_m_inv, cfg.objective.f_eff_m) / bt.mm,
    ))
objective_pupil_geometry = lab_schema.ordered_lab_realism_frame(rows)
objective_pupil_geometry.to_csv(out_csv / "objective_pupil_geometry_summary.csv", index=False)
objective_pupil_geometry


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,generation_method,model_level,hardware_status,plane_label,...,objective_NA,objective_f_eff_mm,pupil_radius_mm,pupil_clipped_fraction,qa_status,regime,pupil_radius_formula,pupil_fill_fraction,pupil_fill_ratio,fourier_ring_radius_mm
0,20260603T205637Z,2026-06-03T20:59:17.499427+00:00,1.0.0,general_objective_pupil,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,objective_pupil_plane,...,0.45,4.0,1.8,0.197899,exploratory,general,f_eff_m * NA / immersion_n,0.802101,1.111111,8238.245119
1,20260603T205637Z,2026-06-03T20:59:17.499505+00:00,1.0.0,limits_objective_pupil,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,objective_pupil_plane,...,0.45,4.0,1.8,0.197899,exploratory,limits,f_eff_m * NA / immersion_n,0.802101,1.111111,37072.103037


In [3]:
rows = []
for regime in ("general", "limits"):
    cfg = vbb_regime.config_for_regime(base, regime)
    design = bt.compute_design_from_targets(cfg.laser, cfg.target, cfg.material)
    grid = bt.make_xy_grid(int(cfg.grid.N), float(cfg.slm.pixel_pitch_m) * float(cfg.grid.device_downsample))
    for blaze_px in (12, 20, 32):
        test_cfg = replace(cfg, slm=replace(cfg.slm, blaze_period_px=blaze_px))
        geom = bt.first_order_filter_geometry(grid, test_cfg.slm, design)
        status = "current_lab_realizable" if geom["first_order_geometry_valid"] else "diagnostic_only"
        row = _common(
            f"{regime}_blaze{blaze_px}",
            test_cfg,
            "fourier_filter_plane",
            "fourier_filter_plane_spatial_frequency_lpmm",
            regime=regime,
            blaze_period_px=blaze_px,
            **geom,
        )
        row["hardware_status"] = status
        rows.append(row)
first_order_filter_geometry = lab_schema.ordered_lab_realism_frame(rows)
first_order_filter_geometry.to_csv(out_csv / "first_order_filter_geometry_summary.csv", index=False)
first_order_filter_geometry


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,generation_method,model_level,hardware_status,plane_label,...,regime,blaze_period_px,configured_filter_radius_lpmm,recommended_filter_radius_lpmm,effective_filter_radius_lpmm,carrier_lpmm,axicon_cone_radius_lpmm,frequency_bin_lpmm,first_order_geometry_valid,first_order_geometry_margin_lpmm
0,20260603T205637Z,2026-06-03T20:59:17.527027+00:00,1.0.0,general_blaze12,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,fourier_filter_plane,...,general,12,2.5,2.500000,2.500000,10.416667,2.059561,0.061035,True,7.916667
1,20260603T205637Z,2026-06-03T20:59:17.527071+00:00,1.0.0,general_blaze20,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,fourier_filter_plane,...,general,20,2.5,2.500000,2.500000,6.250000,2.059561,0.061035,True,3.750000
2,20260603T205637Z,2026-06-03T20:59:17.527093+00:00,1.0.0,general_blaze32,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,fourier_filter_plane,...,general,32,2.5,2.500000,2.500000,3.906250,2.059561,0.061035,True,1.406250
3,20260603T205637Z,2026-06-03T20:59:17.532909+00:00,1.0.0,limits_blaze12,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,fourier_filter_plane,...,limits,12,2.5,9.390096,9.390096,10.416667,9.268026,0.061035,True,1.026571
4,20260603T205637Z,2026-06-03T20:59:17.532947+00:00,1.0.0,limits_blaze20,fast,geometry_only,objective_pupil_limited,hardware_route,diagnostic_only,fourier_filter_plane,...,limits,20,2.5,9.390096,5.937500,6.250000,9.268026,0.061035,False,-3.140096
5,20260603T205637Z,2026-06-03T20:59:17.532967+00:00,1.0.0,limits_blaze32,fast,geometry_only,objective_pupil_limited,hardware_route,diagnostic_only,fourier_filter_plane,...,limits,32,2.5,9.390096,3.710938,3.906250,9.268026,0.061035,False,-5.483846


In [4]:
rows = []
for beam_radius_mm in (1.0, 2.0, 3.0):
    cfg = replace(base, laser=replace(base.laser, beam_radius_on_slm_m=beam_radius_mm * bt.mm))
    pupil_radius = cfg.objective.pupil_radius_m
    fill = objp.gaussian_pupil_fill_fraction(cfg.laser.beam_radius_on_slm_m, pupil_radius)
    rows.append(_common(
        f"beam_radius_{beam_radius_mm:g}mm",
        cfg,
        "objective_pupil_plane",
        "objective_pupil_plane_mm",
        beam_radius_on_slm_mm=beam_radius_mm,
        pupil_fill_fraction=fill,
        pupil_clipped_fraction=1.0 - fill,
        pupil_fill_ratio=objp.pupil_fill_ratio(cfg.laser.beam_radius_on_slm_m, pupil_radius),
    ))
pupil_clipping_summary = lab_schema.ordered_lab_realism_frame(rows)
pupil_clipping_summary.to_csv(out_csv / "pupil_clipping_summary.csv", index=False)
pupil_clipping_summary


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,generation_method,model_level,hardware_status,plane_label,coordinate_frame,objective_NA,objective_f_eff_mm,pupil_radius_mm,pupil_clipped_fraction,qa_status,beam_radius_on_slm_mm,pupil_fill_fraction,pupil_fill_ratio
0,20260603T205637Z,2026-06-03T20:59:17.548996+00:00,1.0.0,beam_radius_1mm,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,objective_pupil_plane,objective_pupil_plane_mm,0.45,4.0,1.8,0.001534,exploratory,1.0,0.998466,0.555556
1,20260603T205637Z,2026-06-03T20:59:17.549023+00:00,1.0.0,beam_radius_2mm,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,objective_pupil_plane,objective_pupil_plane_mm,0.45,4.0,1.8,0.197899,exploratory,2.0,0.802101,1.111111
2,20260603T205637Z,2026-06-03T20:59:17.549039+00:00,1.0.0,beam_radius_3mm,fast,geometry_only,objective_pupil_limited,hardware_route,current_lab_realizable,objective_pupil_plane,objective_pupil_plane_mm,0.45,4.0,1.8,0.486752,exploratory,3.0,0.513248,1.666667
